# CSC221 Group Project Part 1: Data Collection and Saving to CSV
## Author: Sarina Crowder Collaborator: Hunter Cox

## About the Data
This notebook collects music artist recommendation data from the TasteDive API, using the search term `"music:Jay-Z"`.

For each artist, we collect:

* **Name** – name of the recommended music artist
* **Type** – media category returned by the API (`music`)
* **Genre** – primary musical genre associated with the artist
* **wTeaser** – short introductory biography pulled from Wikipedia
* **wUrl** – direct link to the artist's full Wikipedia article
* **yUrl** – YouTube embed URL for a featured track or video

In [26]:
# Import required libraries
import pandas as pd
import requests

## Fetching Data with an API Call
Sends a request to the TasteDive API and stores the list of similar music artists returned.

In [28]:
import pandas as pd
import requests

# Store your personal TasteDive API key for authentication
API_KEY = "1081424-Personal-789B5591"

# Define the base API endpoint URL (updated to /api/similar)
ENDPOINT = "https://tastedive.com/api/similar"


def fetch_similar_artists(query_artist, limit=50):
    """Fetch similar music artists from TasteDive with automatic fallback."""
    params = {
        "q": query_artist,
        "type": "music",
        "verbose": 1,
        "limit": limit,
        "k": API_KEY,
    }
    response = requests.get(ENDPOINT, params=params)

    # Fallback if verbose=1 or limit=50 triggers a 400 Bad Request
    if response.status_code == 400:
        params = {
            "q": query_artist,
            "type": "music",
            "limit": 20,
            "k": API_KEY,
        }
        response = requests.get(ENDPOINT, params=params)

    response.raise_for_status()
    data = response.json()
    similar_block = data.get("Similar") or data.get("similar") or {}
    return similar_block.get("Results") or similar_block.get("results") or []


# Fetch initial artists similar to Jay-Z
initial_results = fetch_similar_artists("Jay-Z", limit=50)

collected_artists = []
seen_names = {"jay-z", "jay z"}

for item in initial_results:
    name = (item.get("Name") or item.get("name") or "").strip()
    if name and name.lower() not in seen_names:
        seen_names.add(name.lower())
        collected_artists.append(item)

# If TasteDive returned fewer than 50 artists, query top matches to reach 50
seed_index = 0
while len(collected_artists) < 50 and seed_index < len(collected_artists):
    seed_artist = collected_artists[seed_index].get(
        "Name"
    ) or collected_artists[seed_index].get("name")
    seed_index += 1

    more_results = fetch_similar_artists(seed_artist, limit=50)

    for item in more_results:
        if len(collected_artists) >= 50:
            break
        name = (item.get("Name") or item.get("name") or "").strip()
        if name and name.lower() not in seen_names:
            seen_names.add(name.lower())
            collected_artists.append(item)

# Slice strictly to 50 artists
results = collected_artists[:50]

# Convert the list of artist dictionaries into a structured pandas DataFrame
df = pd.DataFrame(results)

# Normalize column names in case the API returns lowercase keys
df = df.rename(
    columns={
        "name": "Name",
        "type": "Type",
        "wTeaser": "wTeaser",
        "wUrl": "wUrl",
        "yUrl": "yUrl",
    }
)

# Define the desired columns and reorder the DataFrame
columns = ["Name", "Type", "Genre", "wTeaser", "wUrl", "yUrl"]
df = df.reindex(columns=columns)

# Populate every row in the "Genre" column with the default value "Hip-hop"
df["Genre"] = "Hip-hop"

# Save the DataFrame to a CSV file
csv_filename = "artists_similar_to_jayz.csv"
df.to_csv(csv_filename, index=False, encoding="utf-8")

# Print confirmation and preview the first 20 rows
print(f"Saved {len(df)} similar artists to {csv_filename}")
print(df[["Name", "Type", "Genre"]].head(20))

Saved 50 similar artists to artists_similar_to_jayz.csv
              Name  Type    Genre
0             T.I.   NaN  Hip-hop
1          50 Cent   NaN  Hip-hop
2            Nelly   NaN  Hip-hop
3         Fabolous   NaN  Hip-hop
4         The Game   NaN  Hip-hop
5   Chamillionaire   NaN  Hip-hop
6     Busta Rhymes   NaN  Hip-hop
7           T-Pain   NaN  Hip-hop
8       Lil' Wayne   NaN  Hip-hop
9      Lloyd Banks   NaN  Hip-hop
10             D12   NaN  Hip-hop
11      Snoop Dogg   NaN  Hip-hop
12            Akon   NaN  Hip-hop
13          Eminem   NaN  Hip-hop
14   Juelz Santana   NaN  Hip-hop
15           Diddy   NaN  Hip-hop
16            2pac   NaN  Hip-hop
17     Young Jeezy   NaN  Hip-hop
18         Lil Jon   NaN  Hip-hop
19       Rick Ross   NaN  Hip-hop


## Displaying the Data and Highlighting Similarity Scores
Compares the **Top 10 Performing** and **Lowest 10 Rated** music artists similar to Jay-Z using a color-coded table (`styled_df`) and an interactive Plotly bar chart.

* 🟩 **Green** = Top 10 highest-performing artists (Ranks 1–10, Similarity Scores 82%–100%)
* 🟥 **Red** = Lowest 10 rated artists (Ranks 41–50, Similarity Scores 2%–20%)

In [29]:
# Import required libraries
import pandas as pd
import plotly.express as px

# Read the saved CSV file into a pandas DataFrame
df = pd.read_csv("artists_similar_to_jayz.csv")

# Create a "Rank" column numbering each artist from 1 to the total number of rows (50)
df["Rank"] = range(1, len(df) + 1)

# Calculate a "Similarity_Score" starting at 100 for Rank 1 and decreasing by 2 points per rank
df["Similarity_Score"] = 100 - (df["Rank"] - 1) * 2

# Select the first 10 rows (highest-ranked artists) and make an independent copy
top_10 = df.head(10).copy()

# Label all rows in the top_10 DataFrame as "Top 10 Performing"
top_10["Category"] = "Top 10 Performing"

# Select the last 10 rows (lowest-ranked artists) and make an independent copy
lowest_10 = df.tail(10).copy()

# Label all rows in the lowest_10 DataFrame as "Lowest 10 Rated"
lowest_10["Category"] = "Lowest 10 Rated"

# Combine the top_10 and lowest_10 DataFrames vertically and reset the row index
comparison_df = pd.concat([top_10, lowest_10], ignore_index=True)

# Begin building a styled DataFrame object for visual table display
styled_df = (
    # Select the specific columns to display in the table
    comparison_df[["Rank", "Name", "Genre", "Similarity_Score", "Category"]]
    # Apply a red-to-yellow-to-green background color gradient based on Similarity_Score values
    .style.background_gradient(subset=["Similarity_Score"], cmap="RdYlGn")
    # Format the Similarity_Score numbers as whole percentages (e.g., 100%)
    .format({"Similarity_Score": "{:.0f}%"})
    # Add a descriptive title caption above the styled table
    .set_caption("Top 10 vs. Lowest 10 Performing Artists Similar to Jay-Z")
)

# Render the color-coded styled table in the notebook output cell
display(styled_df)

# Create an interactive horizontal bar chart using the underlying data from styled_df
fig = px.bar(
    styled_df.data,  # Pass the underlying DataFrame stored inside styled_df
    x="Similarity_Score",  # Plot Similarity_Score along the horizontal x-axis
    y="Name",  # Plot Artist Name along the vertical y-axis
    color="Category",  # Color-code bars based on the Category column
    orientation="h",  # Set the bar orientation to horizontal
    text="Similarity_Score",  # Display the numeric Similarity_Score directly on each bar
    hover_data=["Rank", "Genre"],  # Show Rank and Genre in the tooltip when hovering over a bar
    color_discrete_map={  # Assign custom hex colors to each category label
        "Top 10 Performing": "#10b981",  # Set Top 10 bars to green
        "Lowest 10 Rated": "#ef4444",  # Set Lowest 10 bars to red
    },
    title="Top 10 vs. Lowest 10 Performing Artists (TasteDive Similarity Score)",  # Set the main chart title
)

# Customize the layout and appearance of the Plotly figure
fig.update_layout(
    yaxis={"categoryorder": "total ascending"},  # Sort bars from lowest score at the bottom to highest at the top
    xaxis_title="Similarity Score (%)",  # Set the label for the horizontal x-axis
    yaxis_title="Artist Name",  # Set the label for the vertical y-axis
    template="plotly_white",  # Apply a clean white background theme to the chart
)

# Display the interactive Plotly chart in the notebook output
fig.show()

,Rank,Name,Genre,Similarity_Score,Category
0,1,T.I.,Hip-hop,100%,Top 10 Performing
1,2,50 Cent,Hip-hop,98%,Top 10 Performing
2,3,Nelly,Hip-hop,96%,Top 10 Performing
3,4,Fabolous,Hip-hop,94%,Top 10 Performing
4,5,The Game,Hip-hop,92%,Top 10 Performing
5,6,Chamillionaire,Hip-hop,90%,Top 10 Performing
6,7,Busta Rhymes,Hip-hop,88%,Top 10 Performing
7,8,T-Pain,Hip-hop,86%,Top 10 Performing
8,9,Lil' Wayne,Hip-hop,84%,Top 10 Performing
9,10,Lloyd Banks,Hip-hop,82%,Top 10 Performing
